In [23]:
from datetime import datetime
import requests
import pandas as pd
import numpy as np
import time
# --- HÀM TẢI DỮ LIỆU TỪ BINANCE ---
def str_to_millis(date_str):
    dt = datetime.strptime(date_str, "%d/%m/%Y")
    return int(dt.timestamp() * 1000)

def fetch_binance_kline_df(symbol, interval, start_str, end_str):
    base_url = "https://api.binance.us/"
    api_url = "api/v3/klines"
    general_url = base_url + api_url

    start_time = str_to_millis(start_str)
    end_dt = datetime.strptime(end_str, "%d/%m/%Y").replace(hour=23, minute=59, second=59)
    end_time = int(end_dt.timestamp() * 1000)

    all_klines = []
    while start_time < end_time:
        params = {
            "symbol": symbol.upper(),
            "interval": interval,
            "startTime": start_time,
            "endTime": end_time,
            "limit": 1000
        }
        response = requests.get(general_url, params=params)
        if response.status_code != 200:
            print("Error:", response.status_code, response.text)
            break

        data = response.json()
        if not data:
            break
        all_klines.extend(data)

        last_ts = data[-1][0]
        start_time = last_ts + 1
        time.sleep(0.3)

    df = pd.DataFrame(all_klines, columns=[
        "timestamp", "open", "high", "low", "close", "volume",
        "close_time", "quote_asset_volume", "num_trades",
        "taker_buy_base_volume", "taker_buy_quote_volume", "ignore"
    ])
    df = df[["timestamp", "open", "close", "high", "low", "volume"]]
    df.columns = ["timestamp", "open", "close", "high", "low", "volume"]
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df[["open", "close", "high", "low", "volume"]] = df[["open", "close", "high", "low", "volume"]].astype(float)
    df["date"] = df["timestamp"].dt.date
    return df


In [24]:
# --- NHẬP THỜI GIAN ---
symbol = "PAXGUSDT"
interval = "15m"
start_date = "01/01/2020"
end_date = "30/06/2025"

print("\U0001F4E5 Đang tải dữ liệu từ Binance...")
original_df_all = fetch_binance_kline_df(symbol, interval, start_date, end_date)
print(f"✅ Dữ liệu đã tải: {len(original_df_all)} dòng, từ {original_df_all['date'].min()} đến {original_df_all['date'].max()}")

# --- TÁCH DỮ LIỆU ---
original_daily_data = dict(tuple(original_df_all.groupby('date')))
original_sorted_dates = sorted(original_daily_data.keys())
if len(original_sorted_dates) < 30:
    raise Exception("❌ Cần ít nhất 30 ngày dữ liệu!")

📥 Đang tải dữ liệu từ Binance...
✅ Dữ liệu đã tải: 166913 dòng, từ 2020-09-25 đến 2025-06-30


In [39]:
import yfinance as yf
data = yf.download("EURUSD=X", start="2025-06-15", end="2025-07-31", interval = '15m')
data = data.droplevel(1, axis=1)
data.rename(columns={'Open': 'open', 'High': 'high', 'Low': 'low', 'Close': 'close', 'Volume': 'volume'}, inplace=True)
data

/tmp/ipython-input-1727848911.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download("EURUSD=X", start="2025-06-15", end="2025-07-31", interval = '15m')
[*********************100%***********************]  1 of 1 completed


Price,close,high,low,open,volume
Datetime,,,,,
2025-06-15 23:00:00+00:00,1.153403,1.154468,1.153270,1.154468,0
2025-06-15 23:15:00+00:00,1.153669,1.153802,1.153136,1.153270,0
2025-06-15 23:30:00+00:00,1.154335,1.154335,1.153802,1.153802,0
2025-06-15 23:45:00+00:00,1.154068,1.154201,1.153935,1.154068,0
2025-06-16 00:00:00+00:00,1.154601,1.154734,1.153935,1.153935,0
...,...,...,...,...,...
2025-07-30 21:45:00+00:00,1.141032,1.141162,1.140901,1.141162,0
2025-07-30 22:00:00+00:00,1.141944,1.141944,1.140901,1.140901,0
2025-07-30 22:15:00+00:00,1.142074,1.142204,1.141813,1.141944,0


In [25]:
import pandas as pd
import numpy as np

def calculate_adx(df, period=14):
    """Calculate ADX for daily data"""
    high = df['high']
    low = df['low']
    close = df['close']

    # Calculate True Range
    tr1 = high - low
    tr2 = abs(high - close.shift())
    tr3 = abs(low - close.shift())
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

    # Calculate Directional Movement
    plus_dm = high.diff()
    minus_dm = low.diff()
    plus_dm[plus_dm < 0] = 0
    minus_dm[minus_dm > 0] = 0
    minus_dm = abs(minus_dm)

    # Smooth the values
    atr = tr.rolling(period).mean()
    plus_di = 100 * plus_dm.rolling(period).mean() / atr
    minus_di = 100 * minus_dm.rolling(period).mean() / atr

    # Calculate DX and ADX
    dx = 100 * abs(plus_di - minus_di) / (plus_di + minus_di)
    adx = dx.rolling(period).mean()

    return adx

def backtest_strategy(df_15m, fast_ma=5, slow_ma=20, adx_threshold=50, trade_fee=0.001):
    """
    Backtest MA crossover strategy when daily ADX > threshold, including trading fees
    Input: 15-minute OHLCV DataFrame with datetime index
    trade_fee: percentage fee per trade (e.g., 0.001 for 0.1%)
    """
    # Resample to daily for ADX calculation
    if 'timestamp' in df_15m.columns:
       df_15m.set_index('timestamp', inplace=True)
    df_daily = df_15m.resample('D').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).dropna()

    # Calculate daily ADX
    df_daily['adx'] = calculate_adx(df_daily)

    # Calculate 15-minute MAs
    df_15m['fast_ma'] = df_15m['close'].rolling(fast_ma).mean()
    df_15m['slow_ma'] = df_15m['close'].rolling(slow_ma).mean()

    # Initialize signals and positions
    df_15m['signal'] = 0
    df_15m['position'] = 0

    # Map daily ADX to 15-minute timeframe
    df_15m['date'] = df_15m.index.date
    df_daily['date'] = df_daily.index.date
    df_15m = df_15m.merge(df_daily[['date', 'adx']], on='date', how='left')
    df_15m['adx'] = df_15m['adx'].ffill()

    # Generate trading signals
    df_15m['ma_cross'] = np.where(
        (df_15m['fast_ma'] > df_15m['slow_ma']) &
        (df_15m['fast_ma'].shift(1) <= df_15m['slow_ma'].shift(1)), 1,
        np.where(
            (df_15m['fast_ma'] < df_15m['slow_ma']) &
            (df_15m['fast_ma'].shift(1) >= df_15m['slow_ma'].shift(1)), -1, 0
        )
    )

    # Apply ADX filter
    df_15m['signal'] = np.where(df_15m['adx'] > adx_threshold, df_15m['ma_cross'], 0)

    # Calculate positions
    df_15m['position'] = df_15m['signal'].replace(0, np.nan).ffill().fillna(0)

    # Calculate fees (applied on position changes)
    df_15m['trade'] = df_15m['position'].diff().abs()
    df_15m['fee'] = df_15m['trade'] * trade_fee

    # Calculate returns
    df_15m['returns'] = df_15m['close'].pct_change()
    df_15m['strategy_returns'] = df_15m['position'].shift(1) * df_15m['returns'] - df_15m['fee']

    # Calculate cumulative returns
    df_15m['cumulative_returns'] = (1 + df_15m['strategy_returns']).cumprod() - 1

    # Calculate performance metrics
    total_return = df_15m['cumulative_returns'].iloc[-1]
    trades = len(df_15m[df_15m['signal'] != 0])
    win_rate = len(df_15m[df_15m['strategy_returns'] > 0]) / len(df_15m[df_15m['strategy_returns'] != 0]) if trades > 0 else 0

    return {
        'data': df_15m,
        'total_return': total_return,
        'number_of_trades': trades,
        'win_rate': win_rate,
        'total_fees': df_15m['fee'].sum()
    }

# Example usage

# Assuming you have a pandas DataFrame 'df' with 15-minute OHLCV data
# df should have columns: ['open', 'high', 'low', 'close', 'volume']
# and a datetime index

result = backtest_strategy(original_df_all)
print(f"Total Return: {result['total_return']:.2%}")
print(f"Number of Trades: {result['number_of_trades']}")
print(f"Win Rate: {result['win_rate']:.2%}")

# To access the detailed results
result_df = result['data']


Total Return: -100.00%
Number of Trades: 1978
Win Rate: 45.44%


In [40]:
backtest_strategy(data, 5,20,20)

{'data': Price     close      high       low      open  volume   fast_ma   slow_ma  \
 0      1.153403  1.154468  1.153270  1.154468       0       NaN       NaN   
 1      1.153669  1.153802  1.153136  1.153270       0       NaN       NaN   
 2      1.154335  1.154335  1.153802  1.153802       0       NaN       NaN   
 3      1.154068  1.154201  1.153935  1.154068       0       NaN       NaN   
 4      1.154601  1.154734  1.153935  1.153935       0  1.154015       NaN   
 ...         ...       ...       ...       ...     ...       ...       ...   
 3126   1.141032  1.141162  1.140901  1.141162       0  1.141136  1.144036   
 3127   1.141944  1.141944  1.140901  1.140901       0  1.141396  1.143728   
 3128   1.142074  1.142204  1.141813  1.141944       0  1.141474  1.143433   
 3129   1.142465  1.142465  1.142074  1.142074       0  1.141761  1.143164   
 3130   1.142727  1.142727  1.142335  1.142335       0  1.142048  1.142875   
 
 Price  signal  position        date        adx  ma_cr